# 10 · Routing Audit & ORF5 Extraction Demo

**Purpose.** Two things reviewers will ask for around the *alias-aware routing* novelty:
1. **Routing audit** — prove the router sends records with usable annotation to **direct** extraction and records without to **tblastn** lifting (not just that accuracy is high).
2. **ORF5 use-case demo** — the motivating downstream task: extract ORF5 (GP5) across the *entire* PRRSV set, including records that are unannotated or use a different gene name, and report coverage.

## Inputs

FMD/PRRS/PED annotated query sets + references (routing audit); full PRRSV set (ORF5 demo).

In [ ]:
from pathlib import Path
from copy import deepcopy
import sys

import pandas as pd
import matplotlib.pyplot as plt

# --- anchor ROOT to the repo (folder that contains app/src) ---
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "app" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "app" / "data"

FMD_REF    = DATA / "FMD"  / "FMD_ref_test.gb"
FMD_QUERY  = DATA / "FMD"  / "FMD_100seq_anno.gb"
PRRS_REF   = DATA / "PRRS" / "PRRS_ref_test.gb"
PRRS_QUERY = DATA / "PRRS" / "PRRS_100seq_anno.gb"
PED_REF    = DATA / "PED"  / "PED_ref_1.gb"
PED_QUERY  = DATA / "PED"  / "PED_100seqs.gb"

# real unannotated records already in the repo:
FMD_NOANNO  = DATA / "FMD"  / "FMD_OQ211398_noAnno.gb"
PRRS_NOANNO = DATA / "PRRS" / "PRRS_PP946131_noAnno.gb"

RUN_FULL = False
SAMPLE_N = 10


In [ ]:
UNIT_DIR = ROOT / "app" / "validation" / "06_run_tool_extract"
OUT = UNIT_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

## Setup

In [ ]:
from app.src.io.genbank_parser import load_genbank_records, parse_cds_features
from app.src.alias.gene_alias import apply_alias_to_features
from app.src.features.annotation_strategy import get_strategy
from app.src.features.direct_extractor import direct_extract_with_alias
from app.src.lifting.tblastn_lifter import lift_all_tblastn
from app.validation._shared.validation_utils import load_reference_bundle, lifted_to_rows

## Part A — Routing audit

For each record: what did the router decide, and does that match the record's actual annotation status (how many gene names resolve to canonical via the alias map)?

In [ ]:
DATASETS = {"fmd": (FMD_REF, FMD_QUERY), "prrs": (PRRS_REF, PRRS_QUERY), "ped": (PED_REF, PED_QUERY)}

audit = []
for label, (ref_path, query_path) in DATASETS.items():
    bundle = load_reference_bundle(ref_path)
    for rec in load_genbank_records(query_path):
        feats = apply_alias_to_features(parse_cds_features(rec), bundle["alias_lookup"])
        n_resolved = sum(f.get("name_source") in ("alias", "alias_conflict_resolved") for f in feats)
        strategy, ftype = get_strategy(rec, bundle["alias_lookup"])
        audit.append({"virus": label, "record_id": rec.id,
                      "n_cds": len(feats), "n_resolved_names": n_resolved,
                      "has_useful_annotation": n_resolved > 0,
                      "routed_to": strategy, "feature_type": ftype})
audit_df = pd.DataFrame(audit)
audit_df.to_csv(OUT / "routing_audit.tsv", sep="\t", index=False)

# cross-tab: annotation status vs routing decision
crosstab = pd.crosstab(audit_df["has_useful_annotation"], audit_df["routed_to"])
crosstab.to_csv(OUT / "routing_crosstab.tsv", sep="\t")
crosstab

## Expected pattern & consistency check

`has_useful_annotation=True` → `direct`; `False` → `tblastn`. Flag any record that violates this so it can be inspected.

In [ ]:
audit_df["consistent"] = ((audit_df["has_useful_annotation"] & (audit_df["routed_to"] == "direct")) |
                          (~audit_df["has_useful_annotation"] & (audit_df["routed_to"] == "tblastn")))
violations = audit_df[~audit_df["consistent"]]
print(f"{len(violations)} routing inconsistencies out of {len(audit_df)} records")
violations.head(20)

## Part B — ORF5 extraction across the full PRRSV set

The downstream primer-design use case: get ORF5 for **every** record regardless of route, including unannotated / differently-named ones. Coverage = fraction of records for which an ORF5 was produced.

Confirmed from `app/config/prrsv_alias.json`: canonical token is **`ORF5`** (GP5 etc. are aliases that normalise to it).

In [ ]:
ORF5_NAMES = {"ORF5"}   # canonical token (GP5 & variants normalise to ORF5 via the alias map)

bundle = load_reference_bundle(PRRS_REF)
records = load_genbank_records(PRRS_QUERY)

orf5_rows = []
for rec in records:
    strategy, ftype = get_strategy(rec, bundle["alias_lookup"])
    if strategy == "direct":
        lifted = direct_extract_with_alias(rec, ftype, bundle["features"], bundle["alias_lookup"])
    else:
        lifted = lift_all_tblastn(ref_features=bundle["features"], ref_record=bundle["record"],
                                  query_record=rec, validate_codons=(bundle["feature_type"] == "CDS"))
    preds = lifted_to_rows(rec.id, lifted, strategy)
    hit = next((p for p in preds if str(p.get("pred_name")) in ORF5_NAMES), None)
    orf5_rows.append({"record_id": rec.id, "routed_to": strategy,
                      "orf5_found": hit is not None,
                      "start": hit["pred_start"] if hit else None,
                      "end": hit["pred_end"] if hit else None,
                      "coverage": hit.get("coverage") if hit else None})
orf5_df = pd.DataFrame(orf5_rows)
orf5_df.to_csv(OUT / "orf5_extraction.tsv", sep="\t", index=False)

coverage = orf5_df["orf5_found"].mean() * 100
by_route = orf5_df.groupby("routed_to")["orf5_found"].agg(["sum", "size"])
print(f"ORF5 recovered in {orf5_df['orf5_found'].sum()}/{len(orf5_df)} records ({coverage:.1f}%)")
by_route

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
tab = orf5_df.groupby("routed_to")["orf5_found"].mean().mul(100)
tab.plot.bar(ax=ax); ax.set_ylabel("ORF5 recovered %"); ax.set_ylim(0, 100)
ax.set_title("ORF5 extraction coverage by route (PRRSV)")
fig.tight_layout(); fig.savefig(OUT / "orf5_coverage.png", dpi=200)

## Interpretation

> ⚠️ **TODO**: router phân luồng nhất quán với tình trạng annotation; và tool trích được ORF5 cho ~X% toàn tập PRRSV kể cả record không annotation / khác tên — minh họa giá trị end-to-end cho primer design.